# Brain Tumor Segmentation - Google Colab Training
**Train your model on FREE GPU from VS Code!**

How to use:
1. Select Kernel → Google Colab
2. Sign in with Google
3. Run cells one by one (▶️ button)
4. First run takes 5-10 min to setup

## CELL 1: Check GPU & Mount Drive

In [ ]:
# Check GPU availability
import torch
print(f"GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    props = torch.cuda.get_device_properties(0)
    print(f"GPU Memory: {props.total_memory / 1e9:.2f} GB")
else:
    print("GPU not available. Enable in: Runtime → Change runtime type → GPU")

print("\n" + "="*60)
print("Mounting Google Drive...")
print("="*60)

from google.colab import drive
drive.mount('/content/drive', force_remount=True)
print(" Drive mounted!")

GPU Available: True
GPU: Tesla T4
GPU Memory: 15.64 GB

Mounting Google Drive...


## CELL 2: Setup Working Directory

In [18]:
import os

# Create working directory in Drive
work_dir = '/content/drive/MyDrive/brain-tumor-workspace'
os.makedirs(work_dir, exist_ok=True)
os.chdir(work_dir)

print(f"Working directory: {os.getcwd()}")

## CELL 3: Clone Repository & Install Dependencies

In [1]:
print("="*60)
print("STEP 1: Clone Repository")
print("="*60)

import os

# Clone only if not already present
if not os.path.exists('Brain-Tumour-Segmentation'):
    print("Cloning repository...")
    !git clone https://github.com/karthikpagnis/Brain-Tumour-Segmentation.git
    print("Repository cloned!")
else:
    print("Repository already exists")

os.chdir('Brain-Tumour-Segmentation')
print(f"Current directory: {os.getcwd()}")

In [ ]:
print("\n" + "="*60)
print("STEP 2: Install Dependencies (NumPy 2.0+ Compatible)")
print("="*60 + "\n")

print("Installing PyTorch 2.2+ (CUDA 11.8 optimized)...")
!pip install -q --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

print("Installing NumPy 2.0+ (required for modern nibabel)...")
!pip install -q 'numpy>=2.0.0'

print("Installing medical imaging library...")
!pip install -q 'nibabel>=5.1.0'

print("Installing data science libraries...")
!pip install -q scipy pandas scikit-learn

print("Installing visualization libraries...")
!pip install -q matplotlib pillow

print("Installing training & monitoring tools...")
!pip install -q 'pytorch-lightning>=2.2.0' tensorboard tqdm

print("Installing API & utilities...")
!pip install -q fastapi uvicorn pydantic pytest jupyter ipython

print("\n" + "="*60)
print("Verifying installations...")
print("="*60)

import torch
import numpy as np
print(f"✓ PyTorch version: {torch.__version__}")
print(f"✓ NumPy version: {np.__version__}")
print(f"✓ GPU Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✓ GPU Name: {torch.cuda.get_device_name(0)}")

print("\n✓ All dependencies installed successfully!")

## CELL 4: Create Mock Dataset (Quick Test)

In [ ]:
print("="*60)
print("STEP 3: Create Mock BraTS Dataset")
print("="*60 + "\n")

print("Generating synthetic BraTS data (20 cases)...")
print("This will take 1-2 minutes...\n")

!python scripts/download_data.py --create_mock --num_cases 20 --data_dir data/BraTS_MOCK

print("\n✓ Mock dataset created!")
print("  Location: data/BraTS_MOCK/")
print("  Size: ~500MB (20 mock cases)")
print("  Drive space needed after training: ~800MB total")

## CELL 5: Configure for Colab Hardware

In [ ]:
print("="*60)
print("STEP 4: Optimize Config for Colab")
print("="*60 + "\n")

# Read current config
with open('config.py', 'r') as f:
    config = f.read()

# Optimize for free tier GPU
replacements = {
    'BATCH_SIZE = 16': 'BATCH_SIZE = 8',
    'NUM_EPOCHS = 100': 'NUM_EPOCHS = 30',
    'NUM_WORKERS = 4': 'NUM_WORKERS = 2',
    'EARLY_STOPPING_PATIENCE = 20': 'EARLY_STOPPING_PATIENCE = 5',
}

for old, new in replacements.items():
    if old in config:
        config = config.replace(old, new)
        print(f"✓ {old} → {new}")

# Write updated config
with open('config.py', 'w') as f:
    f.write(config)

print("\n✓ Config optimized for Colab T4 GPU!")

## CELL 6: Start Training 🚀

In [ ]:
print("="*60)
print("STEP 5: Start Training on T4 GPU")
print("="*60)
print("\nExpected time: 1-2 hours for 30 epochs on T4 GPU")
print("Batch size: 8 (optimized for free GPU memory)")
print("="*60 + "\n")

import sys
import os

# Ensure Python path is set
current_dir = os.getcwd()
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

!python training/train.py \
    --experiment_name colab_v1 \
    --data_dir data/BraTS_MOCK \
    --epochs 30 \
    --batch_size 8 \
    --device cuda \
    --log_dir outputs/logs

print("\n" + "="*60)
print("✅ TRAINING COMPLETED!")
print("="*60)
print("\nModel checkpoint saved to: checkpoints/colab_v1_best.pth")

## CELL 7: View Training Progress (TensorBoard)

In [ ]:
print("Loading TensorBoard...\n")

%load_ext tensorboard
%tensorboard --logdir outputs/logs/tensorboard

print("\nTensorBoard shows:")
print("  • Loss curves (training & validation)")
print("  • Validation metrics (Dice, IoU, F1)")
print("  • Training progress")

## CELL 8: Save Model to Google Drive

In [ ]:
import shutil
import os

print("="*60)
print("STEP 6: Save Results to Google Drive")
print("="*60 + "\n")

drive_dir = '/content/drive/MyDrive/brain-tumor-workspace'

# Copy best model
if os.path.exists('checkpoints/colab_v1_best.pth'):
    dest = f'{drive_dir}/colab_v1_best.pth'
    shutil.copy('checkpoints/colab_v1_best.pth', dest)
    print(f"✓ Model saved to: {dest}")
else:
    print("⚠️ Model checkpoint not found")

# Copy training summary
if os.path.exists('outputs/colab_v1_summary.json'):
    dest = f'{drive_dir}/colab_v1_summary.json'
    shutil.copy('outputs/colab_v1_summary.json', dest)
    print(f"✓ Summary saved to: {dest}")

# Copy training log
if os.path.exists('outputs/training.log'):
    dest = f'{drive_dir}/training.log'
    shutil.copy('outputs/training.log', dest)
    print(f"✓ Log saved to: {dest}")

print("\n✓ All files saved to Google Drive!")

## CELL 9: Load & Test Model Inference

In [ ]:
print("="*60)
print("STEP 7: Test Model Inference")
print("="*60 + "\n")

import torch
from models.unet_attention import AttentionUNet3D

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}\n")

# Load model
print("Loading model...")
model = AttentionUNet3D().to(device)
checkpoint = torch.load('checkpoints/colab_v1_best.pth', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print("✓ Model loaded!\n")

# Test inference with dummy data
print("Running inference test...")
dummy_input = torch.randn(1, 4, 32, 32, 32).to(device)

with torch.no_grad():
    output = model(dummy_input)

print(f"Input shape:  {dummy_input.shape}")
print(f"Output shape: {output.shape}")
print("✓ Inference test passed!\n")

# Model stats
num_params = sum(p.numel() for p in model.parameters())
print(f"Model Parameters: {num_params / 1e6:.1f}M")
print(f"GPU Memory Used: {torch.cuda.memory_allocated() / 1e9:.2f} GB")

## CELL 10: View Training Results Summary

In [ ]:
import json
import os

print("="*60)
print("TRAINING RESULTS SUMMARY")
print("="*60 + "\n")

summary_file = 'outputs/colab_v1_summary.json'

if os.path.exists(summary_file):
    with open(summary_file, 'r') as f:
        summary = json.load(f)
    
    print(f"Epochs Trained:       {summary.get('epoch', 'N/A')}")
    
    if 'train_loss' in summary and summary['train_loss']:
        print(f"Final Train Loss:     {summary['train_loss'][-1]:.4f}")
    
    if 'val_loss' in summary and summary['val_loss']:
        print(f"Final Val Loss:       {summary['val_loss'][-1]:.4f}")
    
    if 'best_dice' in summary:
        print(f"Best Val Dice:        {summary['best_dice']:.4f}")
    
    if 'best_iou' in summary:
        print(f"Best Val IoU:         {summary['best_iou']:.4f}")
    
    if 'best_f1' in summary:
        print(f"Best Val F1:          {summary['best_f1']:.4f}")
    
    print("\n" + "="*60)
    print("✅ YOU'RE ALL SET!")
    print("="*60)
    print("\nNext steps:")
    print("1. Download model from Google Drive")
    print("2. Run inference: python api/main.py")
    print("3. Try web UI: cd ui && npm install && npm start")
    print("4. See docs/ for deployment options")
else:
    print("⚠️ Summary file not found. Training may still be running.")

## CELL 11: Download Results to Local Machine

In [ ]:
from google.colab import files
import os

print("="*60)
print("DOWNLOAD RESULTS TO LOCAL MACHINE")
print("="*60 + "\n")

# Files to download
download_files = [
    'checkpoints/colab_v1_best.pth',
    'outputs/colab_v1_summary.json',
    'outputs/training.log'
]

print("Preparing files for download...\n")

for file in download_files:
    if os.path.exists(file):
        print(f"Downloading: {file}")
        files.download(file)
    else:
        print(f"⚠️ {file} not found")

print("\n✓ Files downloaded! Check your Downloads folder.")

## Notes

✅ **How to use this notebook:**
1. Click kernel selector (top right) → Select **Google Colab**
2. Sign in with your Google account
3. Run cells one by one (▶️ button or Ctrl+Enter)
4. First run: 5-10 min for setup
5. Training: 1-2 hours on T4 GPU

⚠️ **Important:**
- Replace `YOUR_USERNAME` in CELL 3 with your actual GitHub username
- Enable GPU: Runtime → Change runtime type → GPU
- Model auto-saves to Google Drive

❓ **Issues?**
- GPU timeout? Checkpoints save automatically - run CELL 8 to save
- Out of memory? Reduce BATCH_SIZE in config.py
- Drive full? Check quota and clear space